In [0]:
CREATE OR REPLACE PROCEDURE
dbw_healthcare_de_dev.etl.usp_promote_patients_to_silver(
    IN p_batch_id STRING
)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN

    -- Step 1: Ensure Silver schema exists
    CREATE SCHEMA IF NOT EXISTS
        dbw_healthcare_de_dev.silver;

    -- Step 2: Create an empty Silver table
    -- using the Bronze table schema
    CREATE TABLE IF NOT EXISTS
        dbw_healthcare_de_dev.silver.patients
    LIKE
        dbw_healthcare_de_dev.bronze.patients_raw
    USING DELTA;

    -- Step 3: Merge incoming patients into Silver
    MERGE INTO dbw_healthcare_de_dev.silver.patients AS target
    USING (
        SELECT
            patient_id,
            first_name,
            last_name,
            gender,
            date_of_birth,
            state,
            batch_id,
            source_file,
            ingestion_timestamp
        FROM dbw_healthcare_de_dev.bronze.patients_raw
        WHERE batch_id = p_batch_id
          AND patient_id IS NOT NULL
    ) AS source

    ON target.patient_id = source.patient_id

    WHEN MATCHED AND (
        NOT (target.first_name <=> source.first_name)
        OR NOT (target.last_name <=> source.last_name)
        OR NOT (target.gender <=> source.gender)
        OR NOT (target.date_of_birth <=> source.date_of_birth)
        OR NOT (target.state <=> source.state)
    )
    THEN UPDATE SET
        target.first_name = source.first_name,
        target.last_name = source.last_name,
        target.gender = source.gender,
        target.date_of_birth = source.date_of_birth,
        target.state = source.state,
        target.batch_id = source.batch_id,
        target.source_file = source.source_file,
        target.ingestion_timestamp = source.ingestion_timestamp

    WHEN NOT MATCHED
    THEN INSERT *;

END;



/*
CREATE TABLE IF NOT EXISTS
    dbw_healthcare_de_dev.silver.patients_backup
DEEP CLONE
    dbw_healthcare_de_dev.silver.patients;
*/

SELECT COUNT(*) AS backup_record_count
FROM dbw_healthcare_de_dev.silver.patients_backup;